# TACTIQ — Indexing Pipeline, qadamma-qadam

**Document Loading → Text Cleaning → Metadata → Chunking → Embedding Generation → Qdrant Storage**

Bu notebook `scripts/ingest_documents.py` bajaradigan ishni bosqichma-bosqich ko'rsatadi:
har bir bosqichdan keyin natijani ko'z bilan tekshiramiz.

Ishga tushirishdan oldin (loyiha ildizida):
```bash
pip install -r requirements.txt
docker compose up -d qdrant     # Docker bo'lmasa — pastdagi 6-qadam in-memory rejimga o'tadi
```

In [ ]:
import sys
from pathlib import Path

# Notebook notebooks/ papkasida turibdi. Loyiha ildizi va src/ ni import yo'liga qo'shamiz.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from config.settings import get_settings

settings = get_settings()
DATA_DIR = PROJECT_ROOT / "data" / "sample_docs"
print("Data:", DATA_DIR)
print("Model:", settings.embedding_model_name)
print("Chunk:", settings.chunk_size, "/", settings.chunk_overlap)

## 1. Document Loading

`load_directory()` papkadagi fayllarni rekursiv topadi va har birini `Document(text, metadata)` ga aylantiradi.
PDF'ning har bir sahifasi — alohida `Document`.

In [ ]:
from tactiq.indexing.document_loading import load_directory

load_result = load_directory(DATA_DIR)
print("Yuklangan fayllar:", load_result.loaded_files)
print("Xato bergan:", load_result.failed_files)
print("Document'lar soni:", len(load_result.documents))

for doc in load_result.documents:
    print(doc.metadata)

## 2. Text Cleaning

`low_block.txt` ataylab "iflos" yozilgan: ortiqcha bo'sh joylar va bo'sh qatorlar bor. Oldin va keyin solishtiramiz.

In [ ]:
from tactiq.indexing.text_cleaning import clean_document

low_block = next(d for d in load_result.documents if d.metadata["source"] == "low_block.txt")
print("OLDIN:", repr(low_block.text[:200]))
print()
print("KEYIN:", repr(clean_document(low_block).text[:200]))

cleaned_documents = [clean_document(d) for d in load_result.documents]
cleaned_documents = [d for d in cleaned_documents if d.text]  # bo'sh matnlarni tashlaymiz

## 3. Metadata

`document_id` fayl yo'lidan `uuid5` bilan yasaladi — har safar bir xil.
`topic`, `document_type`, `language` — faqat `<fayl>.meta.json` sidecar fayldan. Bo'lmasa — `None`.

In [ ]:
from tactiq.indexing.metadata import attach_metadata

documents = [attach_metadata(d, DATA_DIR) for d in cleaned_documents]
documents[0].metadata

## 4. Chunking

Token nima? Avval tokenizer matnni qanday bo'lishini ko'ramiz, keyin chunk'larga bo'lamiz.

In [ ]:
from tactiq.indexing.chunking import TokenTextSplitter, make_token_windows

splitter = TokenTextSplitter.from_model_name(
    settings.embedding_model_name, settings.chunk_size, settings.chunk_overlap
)
print(splitter.tokenizer.tokenize("Counterpressing after losing the ball"))

# Oyna (window) matematikasi: 1200 token, 500 / 50
print(list(make_token_windows(1200, chunk_size=500, overlap=50)))

In [ ]:
chunks = splitter.split_documents(documents)
print("Chunk'lar soni:", len(chunks))
for chunk in chunks:
    print(f"{chunk.metadata['source']:28} page={chunk.metadata['page']}  "
          f"chunk_index={chunk.metadata['chunk_index']}  tokens={chunk.metadata['token_count']}")

### Mashq: overlap'ni ko'rish

Kichikroq `chunk_size` bilan `build_up_play.md` ni bo'lib, 1-chunk oxiri va 2-chunk boshini solishtiring.

In [ ]:
small_splitter = TokenTextSplitter(splitter.tokenizer, chunk_size=120, chunk_overlap=20)
build_up = next(d for d in documents if d.metadata["source"] == "build_up_play.md")
small_chunks = small_splitter.split_document(build_up)

print(len(small_chunks), "ta chunk")
print("1-chunk OXIRI :", repr(small_chunks[0].text[-120:]))
print("2-chunk BOSHI :", repr(small_chunks[1].text[:120]))

## 5. Embedding Generation

Har bir chunk → vector. Vector o'lchami (dimension) model chiqargan natijadan olinadi.

In [ ]:
from tactiq.indexing.embeddings import EmbeddingModel

embedding_model = EmbeddingModel(settings.embedding_model_name, settings.embedding_batch_size)
print("Dimension:", embedding_model.dimension)
print("max_seq_length:", embedding_model.max_seq_length, "token")

vectors = embedding_model.embed([c.text for c in chunks])
print("vectors.shape:", vectors.shape)
print("Birinchi vector'ning 8 ta soni:", vectors[0][:8])

In [ ]:
# Ma'no yaqinligi: vector'lar normalize qilingan, shuning uchun dot product = cosine similarity
a, b, c = embedding_model.embed(["high press", "aggressive pressing in the opponent's half", "stadium ticket prices"])
print("high press <-> aggressive pressing :", round(float(a @ b), 3))
print("high press <-> ticket prices       :", round(float(a @ c), 3))

## 6. Qdrant Storage

Docker'dagi Qdrant'ga ulanamiz. Server ishlamasa — o'rganish uchun in-memory rejimga o'tamiz
(notebook yopilganda ma'lumot yo'qoladi).

In [ ]:
from qdrant_client import QdrantClient
from tactiq.indexing.vector_store import QdrantStore

try:
    client = QdrantClient(url=settings.qdrant_url, api_key=settings.qdrant_api_key, timeout=5)
    client.get_collections()
    print("Qdrant server:", settings.qdrant_url)
except Exception as error:
    print("Server topilmadi, in-memory rejim:", type(error).__name__)
    client = QdrantClient(":memory:")

store = QdrantStore(client, "tactiq_notebook_demo")
print("Yangi yaratildi:", store.ensure_collection(embedding_model.dimension))
print("Yozildi:", store.upsert_chunks(chunks, vectors, embedding_model.model_name))
print("Collection'dagi point'lar:", store.count())

In [ ]:
# Qayta yozamiz — barqaror ID'lar tufayli soni o'zgarmasligi kerak
store.upsert_chunks(chunks, vectors, embedding_model.model_name)
print("Qayta yozilgandan keyin:", store.count())

# Bitta point'ning payload'ini ko'ramiz (bu qidiruv emas, shunchaki o'qish)
store.peek(limit=1)[0].payload

## Xulosa

Endi Qdrant'da har bir chunk uchun `id + vector + payload` saqlanmoqda.
Keyingi modul (hali amalga oshirilmagan) — **Retrieval**: savolni xuddi shu model bilan embed qilib,
eng yaqin chunk'larni topish.

Demo collection'ni o'chirish: `client.delete_collection("tactiq_notebook_demo")`